# 02 Normalize Patients

This notebook transforms raw `patients.csv` into a normalized patient structure:
- `patient.csv`
- `patient_address.csv`

The address is stored separately and linked by a foreign key to keep the data model normalized.

In [ ]:
import pandas as pd
import numpy as np
import uuid
from pathlib import Path

RAW_DATA_DIR = Path('data/raw')
PROCESSED_DATA_DIR = Path('data/processed')

PATIENTS_FILE = RAW_DATA_DIR / 'patients.csv'

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def normalize_empty_strings(df: pd.DataFrame) -> pd.DataFrame:
    return df.replace(r'^\s*$', np.nan, regex=True)


def normalize_gender(value: str) -> str:
    if pd.isna(value):
        return 'UNKNOWN'

    value = str(value).strip().upper()

    if value == 'M':
        return 'MALE'
    if value == 'F':
        return 'FEMALE'

    return 'UNKNOWN'


def generate_patient_number(index: int) -> str:
    return f'P{index:08d}'


def clean_string_columns(df: pd.DataFrame) -> pd.DataFrame:
    string_columns = df.select_dtypes(include=['object', 'string']).columns

    for col in string_columns:
        df[col] = df[col].astype('string').str.strip()

    return df

In [ ]:
raw_patients_df = pd.read_csv(PATIENTS_FILE)
print('Raw patients shape:', raw_patients_df.shape)
raw_patients_df.head()

In [ ]:
rename_map = {
    'Id': 'source_patient_id',
    'BIRTHDATE': 'birth_date',
    'DEATHDATE': 'death_date',
    'SSN': 'ssn',
    'DRIVERS': 'drivers_license',
    'PASSPORT': 'passport',
    'PREFIX': 'prefix',
    'FIRST': 'first_name',
    'LAST': 'last_name',
    'SUFFIX': 'suffix',
    'MAIDEN': 'maiden_name',
    'MARITAL': 'marital_status',
    'RACE': 'race',
    'ETHNICITY': 'ethnicity',
    'GENDER': 'gender',
    'BIRTHPLACE': 'birth_place',
    'ADDRESS': 'address_line',
    'CITY': 'city',
    'STATE': 'state',
    'COUNTY': 'county',
    'ZIP': 'zip_code',
    'LAT': 'latitude',
    'LON': 'longitude',
    'HEALTHCARE_EXPENSES': 'healthcare_expenses',
    'HEALTHCARE_COVERAGE': 'healthcare_coverage',
}

patients_df = raw_patients_df.rename(columns=rename_map).copy()
patients_df.head()

In [ ]:
required_columns = [
    'source_patient_id',
    'birth_date',
    'death_date',
    'first_name',
    'last_name',
    'marital_status',
    'race',
    'ethnicity',
    'gender',
    'address_line',
    'city',
    'state',
    'county',
    'zip_code',
]

existing_columns = [col for col in required_columns if col in patients_df.columns]
patients_df = patients_df[existing_columns].copy()

print('Selected columns:', patients_df.columns.tolist())
patients_df.head()

In [ ]:
patients_df = normalize_empty_strings(patients_df)
patients_df = clean_string_columns(patients_df)

if 'birth_date' in patients_df.columns:
    patients_df['birth_date'] = pd.to_datetime(patients_df['birth_date'], errors='coerce').dt.date

if 'death_date' in patients_df.columns:
    patients_df['death_date'] = pd.to_datetime(patients_df['death_date'], errors='coerce').dt.date

if 'first_name' in patients_df.columns:
    patients_df['first_name'] = patients_df['first_name'].str.title()

if 'last_name' in patients_df.columns:
    patients_df['last_name'] = patients_df['last_name'].str.title()

if 'gender' in patients_df.columns:
    patients_df['gender'] = patients_df['gender'].apply(normalize_gender)
else:
    patients_df['gender'] = 'UNKNOWN'

patients_df['deceased'] = patients_df['death_date'].notna()

patients_df['full_name'] = (patients_df['first_name'].fillna('') + ' ' + patients_df['last_name'].fillna('')).str.strip()
patients_df['full_name'] = patients_df['full_name'].replace('', pd.NA)

patients_df = patients_df.drop_duplicates(subset=['source_patient_id']).reset_index(drop=True)

print('Cleaned patients shape:', patients_df.shape)
patients_df.head()

In [ ]:
patient_df = patients_df[[
    'source_patient_id',
    'first_name',
    'last_name',
    'full_name',
    'birth_date',
    'gender',
    'deceased',
    'death_date',
    'marital_status',
    'race',
    'ethnicity',
]].copy()

patient_df.insert(0, 'id', [str(uuid.uuid4()) for _ in range(len(patient_df))])
patient_df.insert(1, 'patient_number', [generate_patient_number(i + 1) for i in range(len(patient_df))])

patient_df.head()

In [ ]:
patient_address_df = patients_df[[
    'source_patient_id',
    'address_line',
    'city',
    'state',
    'county',
    'zip_code',
]].copy()

patient_address_df = patient_address_df.merge(
    patient_df[['id', 'source_patient_id']],
    on='source_patient_id',
    how='inner'
)

patient_address_df = patient_address_df.rename(columns={'id': 'patient_id'})
patient_address_df.insert(0, 'id', [str(uuid.uuid4()) for _ in range(len(patient_address_df))])

patient_address_df = patient_address_df[[
    'id',
    'patient_id',
    'address_line',
    'city',
    'state',
    'county',
    'zip_code',
]].copy()

address_content_columns = ['address_line', 'city', 'state', 'county', 'zip_code']
patient_address_df = patient_address_df[patient_address_df[address_content_columns].notna().any(axis=1)].reset_index(drop=True)

patient_address_df.head()

In [ ]:
print('Patient table shape:', patient_df.shape)
print('Patient address table shape:', patient_address_df.shape)

print('\nPatient null counts:')
print(patient_df.isna().sum())

print('\nPatient address null counts:')
print(patient_address_df.isna().sum())

print('\nDuplicate source_patient_id in patient table:', patient_df['source_patient_id'].duplicated().sum())
print('Duplicate patient_number in patient table:', patient_df['patient_number'].duplicated().sum())
print('Address records without valid patient_id:', patient_address_df['patient_id'].isna().sum())

In [ ]:
patient_output_file = PROCESSED_DATA_DIR / 'patient.csv'
patient_address_output_file = PROCESSED_DATA_DIR / 'patient_address.csv'

patient_df.to_csv(patient_output_file, index=False)
patient_address_df.to_csv(patient_address_output_file, index=False)

print('Exported:', patient_output_file)
print('Exported:', patient_address_output_file)

In [ ]:
display(patient_df.head(10))
display(patient_address_df.head(10))